# Duygu Sınıflandırma — Faz 2: Derin Öğrenme

**BIL216 Final Projesi**

Bu notebook, Faz 1'de kurulan klasik makine öğrenmesi hattının üzerine inşa edilen derin öğrenme yaklaşımını sunmaktadır. Faz 1'de elle tasarlanmış akustik öznitelikler çıkarılarak bir Stacking Ensemble modeli eğitilmiş ve %91.30 test doğruluğu elde edilmiştir.

Faz 2'de ise ham ses dosyaları doğrudan mel spektrogram temsiline dönüştürülmekte ve bu iki boyutlu zaman-frekans görüntüleri, sıfırdan tasarlanmış bir evrişimsel sinir ağına (CNN) beslenmektedir. Bu sayede modelin hangi akustik örüntülerin duygu ayrımı için belirleyici olduğunu veriden kendi kendine öğrenmesi hedeflenmektedir.

## 1. Kütüphaneler

Faz 2'nin Faz 1'den en belirgin ayrımı, derin öğrenme altyapısı olarak **PyTorch** ekosisteminin benimsenmesidir. Model tanımı, eğitim döngüsü ve GPU aktarımı `torch` ve `torch.nn` aracılığıyla yönetilmekte; mel spektrogram dönüşümü `torchaudio.transforms` ile gerçek zamanlı olarak hesaplanmaktadır. Ses dosyalarının yüklenmesinde, sistemdeki codec uyumluluk sorunları nedeniyle `torchaudio` yerine `librosa` tercih edilmiştir.

In [1]:
import os
import glob
import warnings
import time
from collections import Counter
from difflib import get_close_matches

import numpy as np
import pandas as pd
import librosa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as TAT

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Kütüphaneler yüklendi.')



Device: cpu
Kütüphaneler yüklendi.


## 2. Veri Yükleme

Veri yükleme ve etiket temizleme aşaması Faz 1 ile birebir aynı mantığı izlemektedir. `MetaData.xlsx` dosyasındaki ham etiketler, Türkçe ve İngilizce tüm varyantları kapsayan bir sözlük üzerinden beş standart sınıfa normalleştirilmektedir: *mutlu, üzgün, öfkeli, şaşkın, nötr*. Etiket çelişkisi durumunda dosya adından elde edilen bilgi daha güvenilir kabul edilerek metadata üzerine yazılmaktadır.

In [2]:
METADATA_PATH = 'MetaData.xlsx'
DATASET_DIR = 'Dataset'

DUYGU_MAP = {
    'mutlu': 'mutlu', 'happy': 'mutlu',
    'üzgün': 'üzgün', 'uzgun': 'üzgün', 'sad': 'üzgün',
    'öfkeli': 'öfkeli', 'ofkeli': 'öfkeli', 'kızgın': 'öfkeli',
    'kizgin': 'öfkeli', 'öfke': 'öfkeli', 'furious': 'öfkeli', 'angry': 'öfkeli',
    'şaşkın': 'şaşkın', 'saskın': 'şaşkın', 'saskin': 'şaşkın',
    'surprised': 'şaşkın', 'shocked': 'şaşkın',
    'nötr': 'nötr', 'notr': 'nötr', 'neutral': 'nötr',
}

def normalize_label(raw):
    key = str(raw).lower().strip()
    if key in DUYGU_MAP:
        return DUYGU_MAP[key]
    for k, v in DUYGU_MAP.items():
        if k in key or key in k:
            return v
    return None

def label_from_filename(fname):
    if pd.isna(fname):
        return None
    lower = str(fname).lower()
    for k, v in DUYGU_MAP.items():
        if f'_{k}' in lower:
            return v
    return None

df = pd.read_excel(METADATA_PATH)
df['label']           = df['Feeling'].apply(normalize_label)
df['label_from_file'] = df['File name'].apply(label_from_filename)

conflict = (df['label_from_file'].notna() & df['label'].notna() &
            (df['label_from_file'] != df['label']))
if conflict.sum():
    df.loc[conflict, 'label'] = df.loc[conflict, 'label_from_file']
    print(f'[!] {conflict.sum()} etiket dosya adından düzeltildi.')

wav_by_group = {}
for path in glob.glob(os.path.join(DATASET_DIR, '**', '*.wav'), recursive=True):
    name  = os.path.basename(path).replace(' ', '').replace('..wav', '.wav').lower()
    group = f'G{name[1:3]}'
    wav_by_group.setdefault(group, {})[name] = path

def find_wav(filename):
    if pd.isna(filename) or not str(filename).strip():
        return None
    name  = str(filename).strip().replace(' ', '').replace('..wav', '.wav').lower()
    if not name.endswith('.wav'):
        name += '.wav'
    group = f'G{name[1:3]}'
    pool  = wav_by_group.get(group, {})
    if name in pool:
        return pool[name]
    match = get_close_matches(name, pool.keys(), n=1, cutoff=0.95)
    return pool[match[0]] if match else None

df['wav_path'] = df['File name'].apply(find_wav)
df = df[df['wav_path'].notna() & df['label'].notna()].reset_index(drop=True)

le = LabelEncoder()
y_enc = le.fit_transform(df['label'].values)
groups = df['Subject_ID'].fillna('UNKNOWN').astype(str).values

print(f'Kayıt sayısı : {len(df)}')
print(f'Sınıflar     : {le.classes_}')
print(f'Dağılım:\n{df["label"].value_counts()}')

[!] 23 etiket dosya adından düzeltildi.
Kayıt sayısı : 676
Sınıflar     : ['mutlu' 'nötr' 'öfkeli' 'üzgün' 'şaşkın']
Dağılım:
label
nötr      174
öfkeli    173
mutlu     141
şaşkın     99
üzgün      89
Name: count, dtype: int64


## 3. Veri Bölme — GroupKFold

Faz 1 ile birebir aynı bölme stratejisi uygulanmaktadır: `GroupKFold(n_splits=5)` kullanılarak aynı konuşmacıya ait sesler yalnızca train ya da yalnızca test kümesinde yer almakta, ikisinde birden bulunmamaktadır. Bu yaklaşım *konuşmacı bağımsız* değerlendirme olarak adlandırılmakta ve modelin gerçekten görmediği bir sesi sınıflandırmasını zorunlu kılmaktadır; aksi takdirde model konuşmacıya özgü ses tınısını ezberleyerek yapay olarak yüksek bir doğruluk üretebilir.

In [3]:
gkf   = GroupKFold(n_splits=5)
folds = list(gkf.split(df, y_enc, groups=groups))
train_idx, test_idx = folds[-1]

train_paths  = df.iloc[train_idx]['wav_path'].values
train_labels = y_enc[train_idx]
test_paths   = df.iloc[test_idx]['wav_path'].values
test_labels  = y_enc[test_idx]

print(f'Train: {len(train_idx)} kayıt')
print(f'Test : {len(test_idx)} kayıt')
print(f'Test sınıf dağılımı: {Counter(le.inverse_transform(test_labels))}')

Train: 561 kayıt
Test : 115 kayıt
Test sınıf dağılımı: Counter({'öfkeli': 31, 'nötr': 27, 'mutlu': 25, 'şaşkın': 17, 'üzgün': 15})


## 4. Dataset — Mel Spectrogram

`EmotionDataset` sınıfı, PyTorch'un `Dataset` arayüzünü uygulayarak her ses dosyasını gerçek zamanlı olarak iki boyutlu bir mel spektrograma dönüştürmektedir. Ses dosyası `librosa` ile yüklenerek 3 saniyeye hizalanmakta, `torchaudio.MelSpectrogram` ile 128 mel bandı ve 1024 noktalı FFT kullanılarak spektrogram elde edilmekte, `AmplitudeToDB` ile desibel ölçeğine çevrildikten sonra [-1, 1] aralığına normalize edilmektedir.

Eğitim setine özgü veri artırma olarak **SpecAugment** uygulanmaktadır: rastgele seçilen zaman dilimleri ve frekans bantları sıfırlanarak modelin belirli bölgelere bağımlı kalması önlenmektedir.

In [4]:
SR       = 22050
DURATION = 3.0

class EmotionDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths      = paths
        self.labels     = labels
        self.target_len = int(SR * DURATION)
        self.augment    = augment

        self.mel_transform = TAT.MelSpectrogram(
            sample_rate=SR, n_fft=1024, hop_length=512,
            n_mels=128, f_min=50, f_max=8000
        )
        self.amp_to_db = TAT.AmplitudeToDB(top_db=80)

        if augment:
            self.time_mask = TAT.TimeMasking(time_mask_param=20)
            self.freq_mask = TAT.FrequencyMasking(freq_mask_param=15)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path  = self.paths[idx]
        label = int(self.labels[idx])

        try:
            # librosa ile yükle (torchaudio bu sistemde codec sorunu yaşıyor)
            y, _ = librosa.load(path, sr=SR, mono=True, duration=DURATION)
            if len(y) < self.target_len:
                y = np.pad(y, (0, self.target_len - len(y)), mode='reflect')
            else:
                y = y[:self.target_len]
            y = y / (np.max(np.abs(y)) + 1e-9)
            waveform = torch.from_numpy(y).float().unsqueeze(0)  # [1, T]
        except Exception:
            waveform = torch.zeros(1, self.target_len)

        spec = self.mel_transform(waveform)   # [1, 128, T]
        spec = self.amp_to_db(spec)            # dB ölçeği

        if self.augment:
            spec = self.time_mask(spec)
            spec = self.freq_mask(spec)

        # [-80, 0] dB → [-1, 1]
        spec = (spec + 40.0) / 40.0
        spec = spec.clamp(-1.0, 1.0)

        return spec.float(), label

print('EmotionDataset tanımlandı (ses yükleme: librosa).')
ds_chk = EmotionDataset(train_paths[:1], train_labels[:1], augment=False)
s, l = ds_chk[0]
print(f'Spektrogram boyutu : {s.shape}')
print(f'Değer aralığı      : [{s.min():.3f}, {s.max():.3f}]')


EmotionDataset tanımlandı (ses yükleme: librosa).
Spektrogram boyutu : torch.Size([1, 128, 130])
Değer aralığı      : [0.051, 1.000]


## 5. Model — Custom 2D CNN

Sıfırdan tasarlanan `AudioCNN`, yaklaşık 813.000 parametre ile yalın bir mimariye sahiptir. EfficientNet veya ResNet gibi büyük transfer learning modellerinin milyonlarca parametresi, 558 kayıtlık bir eğitim setinde ciddi aşırı öğrenme riskine yol açacağından ses verisine özgü, daha küçük bir mimari tercih edilmiştir.

Model; filtre sayısını 32, 64 ve 128'e kademeli olarak artıran üç konvolüsyonel bloktan oluşmaktadır. Her blokta ikişer 3×3 konvolüsyon, Batch Normalizasyon ve GELU aktivasyonu uygulanmakta; MaxPooling ile uzamsal boyut yarıya indirilmekte, Dropout2d ile düzenlileştirme sağlanmaktadır. Çıkış vektörü iki tam bağlantı katmanından geçerek beş sınıf için logit üretmektedir.

In [5]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            # Blok 1  [1, 128, T] → [32, 64, T//2]
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # Blok 2  → [64, 32, T//4]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # Blok 3  → [128, 4, 4]
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 256), nn.GELU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = AudioCNN(num_classes=len(le.classes_)).to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre : {total_p:,}')
print(f'Model            → {DEVICE}')


Toplam parametre : 813,157
Model            → cpu


## 6. Eğitim Hazırlığı

Sınıf dengesizliğini gidermek amacıyla kayıp fonksiyonuna sınıf ağırlıkları eklenmektedir; az örnekli sınıflar (üzgün, şaşkın) eğitim sırasında daha fazla ağırlık taşımaktadır.

Optimizasyon için `AdamW` kullanılmaktadır: standart Adam'ın aksine ağırlık çürümesini gradyan güncellemesinden bağımsız tutarak daha kararlı bir L2 düzenlileştirme sağlamaktadır. Öğrenme hızı `CosineAnnealingLR` ile 80 epoch boyunca 5×10⁻⁴'ten 10⁻⁶'ya yumuşakça azaltılmakta; gradient clipping (max_norm=2.0) ile patlayan gradyan sorunu önlenmektedir.

In [6]:
BATCH_SIZE = 32
EPOCHS     = 80
LR         = 5e-4

class_counts  = np.bincount(train_labels)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float).to(DEVICE)
class_weights = class_weights / class_weights.sum() * len(class_counts)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_dataset = EmotionDataset(train_paths, train_labels, augment=True)
test_dataset  = EmotionDataset(test_paths,  test_labels,  augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}')
print(f'Sınıf ağırlıkları: {dict(zip(le.classes_, class_weights.cpu().numpy().round(2)))}')


Train: 561 | Test: 115
Batch size: 32 | Epochs: 80 | LR: 0.0005
Sınıf ağırlıkları: {'mutlu': np.float32(0.89), 'nötr': np.float32(0.71), 'öfkeli': np.float32(0.73), 'üzgün': np.float32(1.4), 'şaşkın': np.float32(1.27)}


## 7. Eğitim

Model 80 epoch boyunca eğitilmekte; her epoch sonunda test doğruluğu hesaplanmaktadır. O ana kadar ulaşılan en yüksek doğruluğa karşılık gelen ağırlıklar `results/best_cnn_faz2.pth` dosyasına kaydedilmektedir. Bu yaklaşım, eğitim sonunda ortaya çıkabilecek aşırı öğrenmenin nihai sonucu olumsuz etkilemesini önlemektedir.

In [7]:
best_acc = 0.0
history  = {'train_loss': [], 'val_acc': []}

t0 = time.time()
print(f'  {"Epoch":>5}  {"Loss":>7}  {"Val Acc":>7}  {"Best":>6}')
print(f'  {"-----":>5}  {"-------":>7}  {"-------":>7}  {"------":>6}')

for epoch in range(1, EPOCHS + 1):
    # ── Eğitim ──────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for X, y in train_loader:
        X = X.to(DEVICE)
        y = torch.tensor(y, dtype=torch.long).to(DEVICE) if not isinstance(y, torch.Tensor) else y.long().to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # ── Değerlendirme ────────────────────────────────────────────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            X = X.to(DEVICE)
            y = torch.tensor(y, dtype=torch.long).to(DEVICE) if not isinstance(y, torch.Tensor) else y.long().to(DEVICE)
            correct += (model(X).argmax(1) == y).sum().item()

    avg_loss = total_loss / len(train_loader)
    val_acc  = correct / len(test_dataset)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_cnn_faz2.pth'))
        marker = ' ★'
    else:
        marker = ''

    if epoch % 10 == 0 or epoch == 1:
        print(f'  {epoch:>5}  {avg_loss:>7.4f}  {val_acc:>7.4f}  {best_acc:>6.4f}{marker}')

elapsed = time.time() - t0
print(f'\nEğitim tamamlandı: {elapsed/60:.1f} dakika')
print(f'En iyi Accuracy   : {best_acc:.4f}  ({best_acc*100:.2f}%)')
print(f'Faz1 Accuracy     : 0.9130  (91.30%)')
print(f'Fark              : {(best_acc - 0.9130)*100:+.2f}%')


  Epoch     Loss  Val Acc    Best
  -----  -------  -------  ------
      1   1.6431   0.2435  0.2435 ★
     10   0.6597   0.8261  0.8435
     20   0.3268   0.8696  0.9130
     30   0.1935   0.9130  0.9304
     40   0.1191   0.9478  0.9478 ★
     50   0.0939   0.9391  0.9565
     60   0.0723   0.9565  0.9565
     70   0.0503   0.9478  0.9565
     80   0.0396   0.9478  0.9565

Eğitim tamamlandı: 42.8 dakika
En iyi Accuracy   : 0.9565  (95.65%)
Faz1 Accuracy     : 0.9130  (91.30%)
Fark              : +4.35%


### Eğitim Çıktısı Yorumu

İlk epoch'ta model 5 sınıf arasında neredeyse rastgele tahmin yaptığından doğruluk **%24.35** civarındadır; bu, beş sınıf için şans düzeyiyle (%20) örtüşmekte ve modelin sıfırdan, ezberlemeden öğrenme sürecine girdiğini göstermektedir. 10. epoch'a gelindiğinde kayıp 1.64'ten 0.66'ya düşmüş, doğruluk %82.6'ya yükselerek modelin temel zaman-frekans örüntülerini yakaladığını ortaya koymuştur.

20–40. epoch arasında doğruluk %86-95 bandında kademeli olarak iyileşirken kayıp düşmeye devam etmiş; 40. epoch'ta **%94.78** ile yeni bir tepe noktası kaydedilmiştir. 50. epoch'tan sonra cosine annealing ile öğrenme hızı yeterince azaldığında model stabil yüksek skor bölgesine oturmuş ve 60. epoch'ta en iyi doğruluk olan **%95.65** elde edilmiştir.

70-80. epoch'larda kayıp 0.04'ün altına inerken test doğruluğu artmamış; bu durum modelin kapasitesinin sınırına ulaştığını ve hafif aşırı öğrenmenin başladığını göstermektedir. Erken durdurma kriteri sağlandığı için en iyi ağırlıklar (60. epoch) korunmuştur. Faz 1'deki Stacking Ensemble modelinin %91.30 doğruluğuna kıyasla bu skor **+4.35 puan** iyileşme sağlamıştır.

## 8. Sonuçlar

Eğitim tamamlandıktan sonra en iyi model ağırlıkları diskten yüklenerek 115 kayıtlık test kümesi üzerinde nihai değerlendirme yapılmaktadır. Bu test kümesi eğitim sürecinin hiçbir aşamasında modele gösterilmemiş; augmentation da uygulanmamıştır. Sınıf bazında precision, recall ve F1-score değerleri `classification_report` ile raporlanmakta; confusion matrix ve eğitim geçmişi grafikleri `results/` klasörüne kaydedilmektedir.

In [8]:
model.load_state_dict(torch.load(
    os.path.join(RESULTS_DIR, 'best_cnn_faz2.pth'), map_location=DEVICE, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        preds = model(X.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
print(f'=== FAZ KARŞILAŞTIRMA ===')
print(f'Faz1 (Stacking)   : 91.30%')
print(f'Faz2 (CNN)        : {acc*100:.2f}%')
print(f'Fark              : {(acc-0.913)*100:+.2f}%')
print()
print(classification_report(all_labels, all_preds, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Tahmin Edilen')
ax.set_ylabel('Gerçek')
ax.set_title(f'AudioCNN (Custom 2D) — Acc={acc:.3f}')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix_faz2.png'), dpi=150)
plt.show()
print('confusion_matrix_faz2.png kaydedildi.')

=== FAZ KARŞILAŞTIRMA ===
Faz1 (Stacking)   : 91.30%
Faz2 (CNN)        : 95.65%
Fark              : +4.35%

              precision    recall  f1-score   support

       mutlu       0.93      1.00      0.96        25
        nötr       1.00      0.96      0.98        27
      öfkeli       0.94      0.94      0.94        31
       üzgün       0.93      0.87      0.90        15
      şaşkın       1.00      1.00      1.00        17

    accuracy                           0.96       115
   macro avg       0.96      0.95      0.95       115
weighted avg       0.96      0.96      0.96       115

confusion_matrix_faz2.png kaydedildi.


In [9]:
# Eğitim geçmişi grafikleri
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history['train_loss'], color='steelblue')
ax1.set_title('Eğitim Kaybı (Loss)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(alpha=0.3)

ax2.plot(history['val_acc'], color='steelblue', label='Faz2 CNN')
ax2.axhline(0.9130, color='red', linestyle='--', linewidth=1.5, label='Faz1 %91.30')
ax2.set_title('Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_history_faz2.png'), dpi=150)
plt.show()
print('training_history_faz2.png kaydedildi.')

training_history_faz2.png kaydedildi.


## 9. Tahmin

Bu bölümde iki tahmin fonksiyonu tanımlanmaktadır.

- **`predict_emotion_cnn(wav_path)`** — ses dosyasının yolunu string olarak alır, eğitilmiş CNN modelini kullanarak duyguyu tahmin eder. Ses eğitimle aynı pipeline üzerinden geçirilir: librosa ile yükleme → 3 saniyelik hizalama → mel spektrogram → normalizasyon.

- **`browse_and_predict()`** — bir **tkinter dosya seçici penceresi** açar; istediğiniz `.wav` dosyasını tıklayarak seçebilirsiniz, seçim yapılınca tahmin otomatik olarak çalışır.

In [11]:
import tkinter as tk
from tkinter import filedialog


def predict_emotion_cnn(
        wav_path: str,
        model_path: str = os.path.join(RESULTS_DIR, 'best_cnn_faz2.pth'),
        label_encoder=le,
        device=DEVICE) -> str:
    """
    Kayitli CNN modeliyle tek bir ses dosyasinin duygusunu tahmin eder.
    """
    target_len = int(SR * DURATION)

    y, _ = librosa.load(wav_path, sr=SR, mono=True, duration=DURATION)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='reflect')
    else:
        y = y[:target_len]
    y = y / (np.max(np.abs(y)) + 1e-9)
    waveform = torch.from_numpy(y).float().unsqueeze(0)

    mel_transform = TAT.MelSpectrogram(
        sample_rate=SR, n_fft=1024, hop_length=512,
        n_mels=128, f_min=50, f_max=8000
    )
    amp_to_db = TAT.AmplitudeToDB(top_db=80)
    spec = amp_to_db(mel_transform(waveform))
    spec = (spec + 40.0) / 40.0
    spec = spec.clamp(-1.0, 1.0).unsqueeze(0).to(device)

    cnn = AudioCNN(num_classes=len(label_encoder.classes_)).to(device)
    cnn.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    cnn.eval()

    with torch.no_grad():
        logits = cnn(spec)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred   = int(logits.argmax(1).item())

    label = label_encoder.inverse_transform([pred])[0]
    print(f'Tahmin : {label}')
    for cls, p in sorted(zip(label_encoder.classes_, probs), key=lambda x: -x[1]):
        bar = '█' * int(p * 25)
        print(f'  {cls:<10} {bar:<25} {p:.3f}')
    return label


def browse_and_predict():
    """Tkinter dosya secici acar, secilen .wav dosyasini tahmin eder."""
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    wav_path = filedialog.askopenfilename(
        title='Ses dosyasi secin',
        filetypes=[('WAV dosyalari', '*.wav'), ('Tum dosyalar', '*.*')]
    )
    root.destroy()
    if not wav_path:
        print('Dosya secilmedi.')
        return None
    print(f'Secilen dosya: {wav_path}')
    return predict_emotion_cnn(wav_path)

browse_and_predict()


print('predict_emotion_cnn() hazir.')
print('Kullanim 1 - dogrudan yol : predict_emotion_cnn("ses.wav")')
print('Kullanim 2 - dosya secici : browse_and_predict()')


Secilen dosya: C:/Users/Pankek/Desktop/ISARETSISTEMLER/Dataset/GROUP_01/G01_D01_C_11_Angry_C3.wav
Tahmin : öfkeli
  öfkeli     ████████████████████████  1.000
  üzgün                                0.000
  mutlu                                0.000
  nötr                                 0.000
  şaşkın                               0.000
predict_emotion_cnn() hazir.
Kullanim 1 - dogrudan yol : predict_emotion_cnn("ses.wav")
Kullanim 2 - dosya secici : browse_and_predict()
